#Environment Setup & Data Download

In [6]:
import os

print("--- Step 1: Setting up Repository & Datasets ---")

# Clone repository if needed
if not os.path.exists("google-research"):
    !git clone -q https://github.com/google-research/google-research.git

%cd /content/google-research/tide

# Download ETTh1 dataset
!mkdir -p datasets/ETT-small
!wget -q -O datasets/ETT-small/ETTh1.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv

print("Repository cloned and ETTh1 dataset downloaded successfully.")

--- Step 1: Setting up Repository & Datasets ---
/content/google-research/tide
Repository cloned and ETTh1 dataset downloaded successfully.


#Patch Script to 5 Epochs Maximum

In [8]:
# Replace --min_num_epochs with --train_epochs=5 in the shell script
!sed -i 's/--min_num_epochs=5/--train_epochs=5/g' scripts/etth1.sh
# Verify the edit went through
!cat scripts/etth1.sh

#!/bin/bash
# Copyright 2026 The Google Research Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


for horizon in 96 192
  do
    python3 -m train \
    --transform=true \
    --layer_norm=true \
    --holiday=true \
    --dropout_rate=0.5 \
    --batch_size=512 \
    --hidden_size=512 \
    --num_layers=2 \
    --hist_len=720 \
    --pred_len=$horizon \
    --dataset=etth1 \
    --decoder_output_dim=32 \
    --final_decoder_hidden=16 \
    --num_split=1 \
    --learning_rate=0.0009848942

#Execute Patched Benchmark

In [9]:
print("--- Step 2: Running TiDE (5 Epochs per Horizon) ---")
!bash scripts/etth1.sh

--- Step 2: Running TiDE (5 Epochs per Horizon) ---
I0829 14:54:53.290772 136794743362688 train.py:145] Experiment id: pNQKQ1B
/content/google-research/tide/data_loader.py:94: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pd.date_range(
100% 18/18 [04:05<00:00, 13.66s/it]
I0829 14:58:59.323567 136794743362688 data_loader.py:116] Data Shapes: (7, 14400), (25, 17517), (1, 17420), (1, 17420)
I0000 00:00:1788015539.534941   30713 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13653 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0829 14:58:59.609228 136794743362688 train.py:225] Epoch 0
0it [00:00, ?it/s]I0829 14:58:59.674616 136789054445120 data_loader.py:152] Hist len: 720
/usr/local/lib/python3.13/dist-packages/keras/src/optimizers/base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['tide_model/sequential/mlp_residual/dense/kernel

#Extract Metrics into Summary DataFrame

In [11]:
import glob, json
import pandas as pd

print("--- Step 3: Extracting Completed Metric JSONs ---")
rows = []

# Search for saved test_metrics.json files generated across all completed horizons
metric_files = sorted(glob.glob('results/**/test_metrics.json', recursive=True))

for path in metric_files:
    try:
        with open(path) as f:
            metrics = json.load(f)
            # Folder format: dataset_exptID_horizon
            exp_name = path.split('/')[1]
            horizon = exp_name.split('_')[-1]

            rmse = metrics.get('rmse', 0)
            mse = metrics.get('mse', rmse ** 2 if rmse else 0)
            mae = metrics.get('mae', 0)

            rows.append({
                'Horizon': f"Horizon {horizon}",
                'MSE (5 Epochs)': round(mse, 4),
                'MAE (5 Epochs)': round(mae, 4),
                'RMSE (5 Epochs)': round(rmse, 4)
            })
    except Exception:
        continue

df = pd.DataFrame(rows).drop_duplicates(subset=['Horizon'])

print("\n" + "="*20 + " ETTh1 REPRODUCTION RESULTS " + "="*20)
if not df.empty:
    print(df.to_string(index=False))
else:
    print("No test_metrics.json files saved yet. Check Cell 2 execution logs.")
print("="*68 + "\n")

--- Step 3: Extracting Completed Metric JSONs ---

==================== ETTh1 REPRODUCTION RESULTS ====================
    Horizon  MSE (5 Epochs)  MAE (5 Epochs)  RMSE (5 Epochs)
Horizon 336          1.3388          0.8029           1.1571
Horizon 720          1.1565          0.7553           1.0754
Horizon 192          1.7948          0.9411           1.3397
 Horizon 96          1.9550          0.9794           1.3982

